In [24]:
import time
import csv
from datetime import datetime
import pyvisa
import threading


In [3]:
rm = pyvisa.ResourceManager()
rm.list_resources()

('GPIB0::7::INSTR',)

In [6]:
cryo = rm.open_resource('GPIB0::7::INSTR')

print(cryo.query('*IDN?'))

Cryocon Model 34, Rev 4.06A


In [14]:
def connect_cryocon(resource):
    rm = pyvisa.ResourceManager()
    inst = rm.open_resource(resource)
    inst.timeout = 5000
    inst.read_termination = '\n'
    inst.write_termination = '\n'

    print(inst.query("*IDN?"))
    return inst


In [16]:
def read_temp(inst, channel):
    try:
        return float(inst.query(f"INPUT? {channel}").strip())
    except Exception as e:
        print(f"Error reading channel {channel}: {e}")
        return None



In [37]:
class CryoconLogger:
    def __init__(self, inst, channels=("C", "D"), filename="cryocon_log.csv", interval=10):
        self.inst = inst
        self.channels = list(channels)
        self.filename = filename
        self.interval = interval #in seconds
        self.running = False
        self.pending_comment = ""
        with open(self.filename, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", *[f"T_{ch}" for ch in self.channels], "comments"])

    def add_comment(self, text):
        self.pending_comment = str(text).strip()

    def stop(self):
        self.running = False
        if hasattr(self, "thread"):
            self.thread.join()
        print("Logging stopped.")

    def start(self):
        self.thread = threading.Thread(target=self._run)
        self.running = True
        self.thread.start()

    def _run(self):
        while self.running:
            timestamp = datetime.now().isoformat()
            temps = [read_temp(self.inst, channel) for channel in self.channels]
            comments = self.pending_comment
            self.pending_comment = ""

            #print(timestamp, temps, comments)

            with open(self.filename, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, *temps, comments])
                if comments != "":
                    print(f"Logged at {timestamp}:, Comment='{comments}'")

            time.sleep(self.interval)

In [38]:
inst = connect_cryocon("GPIB0::7::INSTR")

logger = CryoconLogger(inst, channels=("C", "D"), filename="testing.csv", interval=10)

logger.start()


Cryocon Model 34, Rev 4.06A


In [39]:
logger.add_comment("Testing")


Logged at 2026-06-12T18:32:04.794022:, Comment='Testing'


In [36]:
logger.stop()

Logging stopped.
